# Evaluación Parcial 1 - Fundamentos de Deep Learning
### Dataset: CIFAR-100 - Clasificación de imágenes con MLP
Este notebook implementa una red MLP para el dataset CIFAR-100, comparando funciones de activación y aplicando callbacks como ModelCheckpoint.


In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import ModelCheckpoint
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd
from tensorflow.keras.callbacks import TensorBoard
import datetime


In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar100.load_data(label_mode='fine')
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
x_train = x_train.reshape((x_train.shape[0], -1))
x_test = x_test.reshape((x_test.shape[0], -1))
y_train = tf.keras.utils.to_categorical(y_train, 100)
y_test = tf.keras.utils.to_categorical(y_test, 100)

169001437/169001437 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step


In [3]:
def crear_modelo(activation):
    model = models.Sequential([
        layers.Dense(512, activation=activation, input_shape=(3072,)),
        layers.Dropout(0.5),
        layers.Dense(256, activation=activation),
        layers.Dropout(0.5),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

def crear_callback_checkpoint(nombre_experimento):
    return ModelCheckpoint(
        filepath=f"{nombre_experimento}.h5",
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )

## Experimento 1: Comparación de funciones de activación (ReLU vs Tanh)
### Objetivo
Evaluar el impacto de distintas funciones de activación sobre el desempeño del modelo.

### Metodología
- Se usan modelos idénticos cambiando solo `ReLU` por `Tanh`.
- Se entrena con 10 épocas, batch de 128, usando checkpoint para guardar el mejor modelo.
- Se comparan gráficamente y con métricas.

### Hipótesis
ReLU es más eficiente en la práctica, pero tanh puede funcionar mejor si los datos son centrados en cero.


In [ ]:
# ReLU
model_relu = crear_modelo('relu')
cb_relu = crear_callback_checkpoint('mejor_relu')
history_relu = model_relu.fit(x_train, y_train, epochs=20, batch_size=128,
                              validation_split=0.2, callbacks=[cb_relu], verbose=0)

# Tanh
model_tanh = crear_modelo('tanh')
cb_tanh = crear_callback_checkpoint('mejor_tanh')
history_tanh = model_tanh.fit(x_train, y_train, epochs=20, batch_size=128,
                              validation_split=0.2, callbacks=[cb_tanh], verbose=0)

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
plt.figure(figsize=(20,5))
plt.plot(history_relu.history['val_accuracy'], label='ReLU - Validación')
plt.plot(history_tanh.history['val_accuracy'], label='Tanh - Validación')
plt.title('Comparación de funciones de activación')
plt.xlabel('Época')
plt.ylabel('Precisión de validación')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
_, acc_relu = model_relu.evaluate(x_test, y_test, verbose=0)
_, acc_tanh = model_tanh.evaluate(x_test, y_test, verbose=0)
print(f"Precisión en test - ReLU: {acc_relu:.4f}")
print(f"Precisión en test - Tanh: {acc_tanh:.4f}")

En este experimento se comparó el rendimiento de dos funciones de activación: ReLU y Tanh. Ambas fueron evaluadas en un modelo MLP entrenado con el dataset CIFAR-100.

Los resultados mostraron que el modelo con Tanh alcanzó una mayor precisión tanto en validación como en test (9.60%), mientras que ReLU solo logró 6.75%.

Esto sugiere que, al menos en este experimento, Tanh permitió una mejor convergencia del modelo. Una posible explicación es que la activación Tanh, al ser simétrica respecto al eje Y y tener salida entre -1 y 1, puede haber favorecido una mejor propagación del gradiente en las capas densas.

## Experimento 2: Comparación con y sin Dropout

### Objetivo
Analizar el impacto de la técnica de regularización **Dropout** en el desempeño del modelo. Dropout apaga aleatoriamente un porcentaje de las neuronas durante el entrenamiento, evitando que el modelo memorice y ayudando a mejorar la generalización.

### Metodología
Se comparan dos modelos:
- **Modelo A**: incluye capas Dropout (probabilidad de 0.5).
- **Modelo B**: no utiliza Dropout.

Ambos modelos se entrenan bajo las mismas condiciones y se aplica **ModelCheckpoint** para guardar el mejor resultado según `val_accuracy`.

### Hipótesis
El modelo con Dropout puede alcanzar mejor rendimiento en test al evitar el sobreajuste, aunque inicialmente pueda aprender más lento.


In [ ]:
# Modelo con Dropout
def crear_modelo_con_dropout():
    model = models.Sequential([
        layers.Dense(512, activation='tanh', input_shape=(3072,)),
        layers.Dropout(0.5),
        layers.Dense(256, activation='tanh'),
        layers.Dropout(0.5),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Modelo sin Dropout
def crear_modelo_sin_dropout():
    model = models.Sequential([
        layers.Dense(512, activation='tanh', input_shape=(3072,)),
        layers.Dense(256, activation='tanh'),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Entrenamiento con Dropout
model_dropout = crear_modelo_con_dropout()
cb_dropout = crear_callback_checkpoint("modelo_con_dropout")
history_dropout = model_dropout.fit(x_train, y_train, epochs=20, batch_size=128,
                                    validation_split=0.2, callbacks=[cb_dropout], verbose=0)

# Entrenamiento sin Dropout
model_sin_dropout = crear_modelo_sin_dropout()
cb_sin_dropout = crear_callback_checkpoint("modelo_sin_dropout")
history_sin_dropout = model_sin_dropout.fit(x_train, y_train, epochs=20, batch_size=128,
                                            validation_split=0.2, callbacks=[cb_sin_dropout], verbose=0)


In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(history_dropout.history['val_accuracy'], label='Con Dropout - Validación')
plt.plot(history_sin_dropout.history['val_accuracy'], label='Sin Dropout - Validación')
plt.title('Comparación con y sin Dropout')
plt.xlabel('Época')
plt.ylabel('Precisión de validación')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
_, acc_dropout = model_dropout.evaluate(x_test, y_test, verbose=0)
_, acc_sin_dropout = model_sin_dropout.evaluate(x_test, y_test, verbose=0)

print(f"Precisión en test - Con Dropout: {acc_dropout:.4f}")
print(f"Precisión en test - Sin Dropout: {acc_sin_dropout:.4f}")


En este experimento se evaluó el impacto del uso de la técnica de regularización Dropout. El modelo sin Dropout alcanzó una precisión en test de 14.69%, mientras que el modelo con Dropout alcanzó solo 9.99%.

La diferencia podría deberse a que el modelo aún no entra en sobreajuste durante las primeras 10 épocas, por lo que Dropout no resulta beneficioso en esta etapa temprana. Sin embargo, se espera que su efecto sea más positivo en entrenamientos más largos o modelos más complejos.

## Experimento 3: Comparación con y sin Batch Normalization

### Objetivo
Evaluar el impacto de la técnica de normalización por lotes (**Batch Normalization**) en la convergencia y precisión del modelo.

### Metodología
Se comparan dos modelos:
- **Modelo A**: incluye capas Batch Normalization después de cada capa densa.
- **Modelo B**: no utiliza Batch Normalization.

Ambos modelos se entrenan bajo las mismas condiciones y se guarda el mejor modelo con ModelCheckpoint.

### Hipótesis
Batch Normalization puede acelerar y estabilizar el aprendizaje, por lo que se espera un mejor desempeño con esta técnica, especialmente en redes profundas.


In [ ]:
# Modelo con Batch Normalization
def crear_modelo_batchnorm():
    model = models.Sequential([
        layers.Dense(512, input_shape=(3072,)),
        layers.BatchNormalization(),
        layers.Activation('tanh'),
        layers.Dense(256),
        layers.BatchNormalization(),
        layers.Activation('tanh'),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Modelo sin Batch Normalization (igual al usado antes)
def crear_modelo_sin_batchnorm():
    model = models.Sequential([
        layers.Dense(512, activation='tanh', input_shape=(3072,)),
        layers.Dense(256, activation='tanh'),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Entrenamiento con Batch Normalization
model_bn = crear_modelo_batchnorm()
cb_bn = crear_callback_checkpoint("modelo_con_batchnorm")
history_bn = model_bn.fit(x_train, y_train, epochs=20, batch_size=128,
                          validation_split=0.2, callbacks=[cb_bn], verbose=0)

# Entrenamiento sin Batch Normalization
model_no_bn = crear_modelo_sin_batchnorm()
cb_no_bn = crear_callback_checkpoint("modelo_sin_batchnorm")
history_no_bn = model_no_bn.fit(x_train, y_train, epochs=20, batch_size=128,
                                validation_split=0.2, callbacks=[cb_no_bn], verbose=0)


In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(history_bn.history['val_accuracy'], label='Con BatchNorm - Validación')
plt.plot(history_no_bn.history['val_accuracy'], label='Sin BatchNorm - Validación')
plt.title('Comparación con y sin Batch Normalization')
plt.xlabel('Época')
plt.ylabel('Precisión de validación')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
_, acc_bn = model_bn.evaluate(x_test, y_test, verbose=0)
_, acc_no_bn = model_no_bn.evaluate(x_test, y_test, verbose=0)

print(f"Precisión en test - Con BatchNorm: {acc_bn:.4f}")
print(f"Precisión en test - Sin BatchNorm: {acc_no_bn:.4f}")


En este experimento se comparó el uso de la técnica de normalización por lotes (Batch Normalization). El modelo que incluía esta técnica alcanzó una precisión en test de 18.31%, superando al modelo sin BatchNorm que alcanzó 14.61%.

Esto confirma que Batch Normalization estabiliza y acelera el entrenamiento, ayudando a mejorar el rendimiento de la red al controlar la distribución de las activaciones y facilitar el flujo del gradiente.



## Experimento 4: Comparación con y sin Regularización L2

### Objetivo
Evaluar el impacto de la regularización L2 en el rendimiento del modelo. Esta técnica penaliza los pesos grandes para evitar el sobreajuste.

### Metodología
Se comparan dos modelos:
- **Modelo A**: con regularización L2 aplicada en las capas densas.
- **Modelo B**: sin regularización.

Ambos modelos se entrenan bajo las mismas condiciones, y se utiliza ModelCheckpoint para guardar el mejor modelo.

### Hipótesis
La regularización L2 puede mejorar la generalización del modelo y evitar que memorice los datos de entrenamiento.


In [ ]:
from tensorflow.keras import regularizers

# Modelo con L2
def crear_modelo_con_l2():
    model = models.Sequential([
        layers.Dense(512, activation='tanh', input_shape=(3072,),
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.Dense(256, activation='tanh',
                     kernel_regularizer=regularizers.l2(0.001)),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Modelo sin L2
def crear_modelo_sin_l2():
    model = models.Sequential([
        layers.Dense(512, activation='tanh', input_shape=(3072,)),
        layers.Dense(256, activation='tanh'),
        layers.Dense(100, activation='softmax')
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Entrenamiento con L2
model_l2 = crear_modelo_con_l2()
cb_l2 = crear_callback_checkpoint("modelo_con_l2")
history_l2 = model_l2.fit(x_train, y_train, epochs=20, batch_size=128,
                          validation_split=0.2, callbacks=[cb_l2], verbose=0)

# Entrenamiento sin L2
model_no_l2 = crear_modelo_sin_l2()
cb_no_l2 = crear_callback_checkpoint("modelo_sin_l2")
history_no_l2 = model_no_l2.fit(x_train, y_train, epochs=20, batch_size=128,
                                validation_split=0.2, callbacks=[cb_no_l2], verbose=0)


In [ ]:
plt.figure(figsize=(20, 5))
plt.plot(history_l2.history['val_accuracy'], label='Con L2 - Validación')
plt.plot(history_no_l2.history['val_accuracy'], label='Sin L2 - Validación')
plt.title('Comparación con y sin Regularización L2')
plt.xlabel('Época')
plt.ylabel('Precisión de validación')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
_, acc_l2 = model_l2.evaluate(x_test, y_test, verbose=0)
_, acc_no_l2 = model_no_l2.evaluate(x_test, y_test, verbose=0)

print(f"Precisión en test - Con L2: {acc_l2:.4f}")
print(f"Precisión en test - Sin L2: {acc_no_l2:.4f}")


En este experimento se evaluó el uso de regularización L2 (weight decay). El modelo sin L2 alcanzó una precisión de 15.23%, ligeramente superior al modelo con L2 que alcanzó 14.85%.

Esto puede deberse a que en las primeras 10 épocas, el modelo aún no presenta signos evidentes de sobreajuste, por lo que la penalización adicional de L2 podría estar dificultando el aprendizaje útil. Se espera que en entrenamientos más largos, la regularización L2 sea más efectiva para evitar el sobreajuste.

## Métricas detalladas: Precision, Recall y F1-score

Además de la métrica `accuracy`, se calcularán métricas más completas para evaluar el rendimiento de los modelos en clasificación multiclase:

- **Precision**: ¿De los que predije como clase X, cuántos eran realmente clase X?
- **Recall**: ¿De los que eran clase X, cuántos logré predecir correctamente?
- **F1-Score**: Media armónica entre precision y recall.


In [ ]:
# Obtener las predicciones
y_pred = model_bn.predict(x_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true = np.argmax(y_test, axis=1)

# Calcular el reporte
report = classification_report(y_true, y_pred_classes, output_dict=True)
report_df = pd.DataFrame(report).transpose()

# Mostrar el reporte como tabla
report_df.head(15)  # puedes mostrar más si quieres


## Comparación de métricas: Accuracy, Precision, Recall y F1-score

Para tener una visión más completa del rendimiento de los modelos, se calcularán las métricas detalladas de clasificación:
- `Precision`: exactitud de las predicciones positivas.
- `Recall`: cobertura de las verdaderas instancias positivas.
- `F1-score`: balance entre precisión y recall.

Esto permitirá entender qué tan equilibrado es el desempeño del modelo más allá del accuracy.


In [ ]:
def evaluar_modelo(nombre, modelo, x_test, y_test):
    y_pred = modelo.predict(x_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true = np.argmax(y_test, axis=1)

    report = classification_report(y_true, y_pred_classes, output_dict=True, zero_division=0)
    df = pd.DataFrame(report).transpose()
    df["modelo"] = nombre
    return df

# Cambia estos modelos si tus "ganadores" son otros:
df_tanh = evaluar_modelo("Tanh", model_tanh, x_test, y_test)
df_sin_dropout = evaluar_modelo("Sin Dropout", model_sin_dropout, x_test, y_test)
df_batchnorm = evaluar_modelo("BatchNorm", model_bn, x_test, y_test)
df_sin_l2 = evaluar_modelo("Sin L2", model_no_l2, x_test, y_test)

# Juntar todo
df_comparacion = pd.concat([df_tanh, df_sin_dropout, df_batchnorm, df_sin_l2])
df_comparacion_agrupado = df_comparacion.loc[["accuracy", "macro avg", "weighted avg"]]

# Mostrar comparativa final
df_comparacion_agrupado[["precision", "recall", "f1-score", "modelo"]]


Modelo con mejor desempeño general: BatchNorm
Accuracy: 18.31%

Macro avg F1-score: 0.1760

Weighted avg F1-score: 0.1760

El modelo con Batch Normalization no solo alcanzó el mejor accuracy, sino que además mantiene un buen equilibrio entre precisión y recall en todas las clases (lo que refleja su F1 alto tanto macro como ponderado).

Modelo más débil: Tanh
Accuracy y F1 muy bajos (≈ 9.59%)

Esto confirma que la arquitectura sola con Tanh no es suficiente para este tipo de tarea en este contexto.

## Integración de TensorBoard

Con el objetivo de visualizar de forma más detallada el comportamiento del modelo durante el entrenamiento, se integra la herramienta **TensorBoard**, que permite monitorear curvas de pérdida y precisión, así como otros aspectos clave del proceso de optimización.

En este experimento se aplicó TensorBoard al modelo con **Batch Normalization**, el cual obtuvo el mejor rendimiento global.


In [ ]:
# Crear carpeta de logs con marca de tiempo
log_dir_bn = "logs/batchnorm/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_bn = TensorBoard(log_dir=log_dir_bn, histogram_freq=1)

# Entrenar el modelo con TensorBoard y ModelCheckpoint
model_bn = crear_modelo_batchnorm()
cb_bn = crear_callback_checkpoint("modelo_con_batchnorm_tensorboard")

history_bn = model_bn.fit(
    x_train, y_train,
    epochs=20,
    batch_size=128,
    validation_split=0.2,
    callbacks=[cb_bn, tensorboard_bn],
    verbose=0
)


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs/batchnorm


En esta gráfica de TensorBoard se ve la evolución del parámetro beta del Batch Normalization en diferentes capas. Se observa que a medida que pasan las épocas, los valores de beta cambian gradualmente, lo cual indica que la red está aprendiendo y ajustando la normalización en cada batch. Este tipo de análisis permite validar visualmente que los parámetros no se estancan ni divergen.

## Carga del mejor modelo guardado con ModelCheckpoint

Una vez entrenado el modelo con `ModelCheckpoint`, se guarda automáticamente el archivo `.h5` con la mejor versión del modelo según la métrica `val_accuracy`.

A continuación, se muestra cómo cargar ese modelo guardado sin necesidad de reentrenar, para usarlo en predicciones o evaluación directa.


In [ ]:
from tensorflow.keras.models import load_model

# Cargar el mejor modelo guardado
modelo_mejor = load_model("modelo_con_batchnorm_tensorboard.h5")

# Evaluar el modelo cargado
loss_mejor, acc_mejor = modelo_mejor.evaluate(x_test, y_test, verbose=0)
print(f"Precisión del modelo cargado desde .h5: {acc_mejor:.4f}")


Se utilizó ModelCheckpoint para guardar automáticamente el mejor modelo según la métrica val_accuracy. Luego, cargué el archivo .h5 con load_model para reutilizar el modelo sin tener que volver a entrenarlo. Esto es útil para producción, análisis adicional o si el entrenamiento es costoso.

In [ ]:
!pip install gradio


In [ ]:
import gradio as gr

# Cargar modelo entrenado
modelo_gradio = load_model("modelo_con_batchnorm_tensorboard.h5")

# Etiquetas CIFAR-100
labels = tf.keras.datasets.cifar100.load_data(label_mode='fine')[1][1]

# Función de predicción
def clasificar_imagen(imagen):
    imagen = imagen.resize((32, 32))
    img_array = np.array(imagen) / 255.0
    img_array = img_array.reshape(1, 32 * 32 * 3)
    pred = modelo_gradio.predict(img_array)
    clase = np.argmax(pred)
    return f"Clase predicha: {clase}"

# Interfaz Gradio
demo = gr.Interface(
    fn=clasificar_imagen,
    inputs=gr.Image(type="pil"),
    outputs="text",
    title="Clasificación CIFAR-100 con MLP",
    description="Carga una imagen (32x32) para predecir su clase con el modelo entrenado."
)

demo.launch()


## Conclusión General

Tras la ejecución de cuatro experimentos y el análisis de métricas como `accuracy`, `precision`, `recall` y `f1-score`, se concluye que:

- El modelo que incluyó **Batch Normalization** fue el más efectivo, obteniendo no solo la mayor precisión en test (18.31%), sino también el mejor equilibrio entre clases, reflejado en sus valores de F1-score macro y ponderado.
- El modelo sin técnicas adicionales pero con función Tanh obtuvo el rendimiento más bajo, confirmando que la arquitectura simple no es suficiente para esta tarea multiclase.
- Las técnicas de regularización como **Dropout** y **L2** mostraron desempeños correctos pero no superiores, sugiriendo que podrían beneficiar más en entrenamientos más extensos o datasets con más riesgo de overfitting.
- El uso de métricas múltiples permitió una evaluación más justa y completa que el accuracy solo, mostrando que no siempre el modelo con mayor precisión global es el más balanceado.

Este análisis evidencia la importancia de ajustar componentes clave en redes neuronales, y demuestra cómo técnicas específicas como BatchNorm pueden marcar una diferencia significativa en el rendimiento de modelos de clasificación profunda.
